# 线性回归算法及其Python实现

本教程将从基础开始介绍线性回归算法，并通过Python代码实现各种线性回归方法。我们将从数据准备开始，一直学习到实际应用案例，包括正则化和优化技术。

## 1. 数据准备

在本节中，我们将导入必要的库并准备用于演示线性回归的数据集，包括真实数据集和人工生成的数据。

In [5]:
# 导入必要的库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 设置显示格式
plt.style.use('seaborn-v0_8')  # 使用seaborn风格，更新为matplotlib 3.6+兼容版本
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_context("notebook", font_scale=1.5)
sns.set_style("whitegrid")  # 单独设置whitegrid样式

# 设置随机种子以确保结果可复现
np.random.seed(42)

### 1.1 创建合成数据集

我们首先创建一个简单的合成数据集，以便理解线性回归的基本概念。

In [ ]:
# 创建简单的一元线性数据集
X_simple = np.linspace(0, 10, 100).reshape(-1, 1)
y_simple = 2 * X_simple.reshape(-1) + 5 + np.random.normal(0, 1, 100)

# 可视化数据
plt.figure(figsize=(10, 6))
plt.scatter(X_simple, y_simple, alpha=0.7)
plt.xlabel('X')
plt.ylabel('y')
plt.title('简单线性关系数据集')
plt.show()

### 1.2 创建多变量数据集

接下来，我们创建一个具有多个特征的数据集。

In [ ]:
# 创建多变量回归数据集
X_multi, y_multi = make_regression(n_samples=200, n_features=5, 
                                  n_informative=3, noise=15, random_state=42)

# 查看数据形状
print(f"X_multi shape: {X_multi.shape}")
print(f"y_multi shape: {y_multi.shape}")

# 将数据转换为DataFrame以便于查看
df_multi = pd.DataFrame(X_multi, columns=[f'Feature_{i+1}' for i in range(X_multi.shape[1])])
df_multi['Target'] = y_multi
print("\n数据集前5行:")
df_multi.head()

### 1.3 加载真实数据集

最后，我们将加载一个真实的数据集 - California Housing 数据集，用于后续的实际应用案例。

In [ ]:
# 加载California Housing数据集
housing = fetch_california_housing()
X_housing = pd.DataFrame(housing.data, columns=housing.feature_names)
y_housing = housing.target

print(f"California Housing数据集形状: {X_housing.shape}")
print(f"特征名称: {housing.feature_names}")

# 查看数据摘要
X_housing.describe()

## 2. 线性回归基础

线性回归是最基本也是最重要的统计学习方法之一。本节介绍线性回归的数学基础，包括模型表达式、损失函数的定义和推导过程。

### 2.1 线性回归模型表达式

线性回归模型假设目标变量y与特征变量x之间存在线性关系：

##### 一元线性回归：
$y = w_0 + w_1 x + \epsilon$

##### 多元线性回归：
$y = w_0 + w_1 x_1 + w_2 x_2 + ... + w_n x_n + \epsilon$

其中：
- $y$ 是目标变量
- $w_0, w_1, ..., w_n$ 是模型参数（权重）
- $x_1, x_2, ..., x_n$ 是特征变量
- $\epsilon$ 是随机误差项

### 2.2 损失函数 - 均方误差(MSE)

线性回归的目标是找到最佳的参数值，使得预测值与真实值之间的差异最小化。最常用的损失函数是均方误差(Mean Squared Error, MSE)：

$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$

其中：
- $y_i$ 是第i个样本的真实值
- $\hat{y}_i$ 是模型对第i个样本的预测值
- $n$ 是样本数量

In [ ]:
# 定义一个函数来计算MSE
def calculate_mse(y_true, y_pred):
    """计算均方误差"""
    return np.mean((y_true - y_pred) ** 2)

# 示例：假设我们有一些真实值和预测值
y_true = np.array([3, 5, 7, 9, 11])
y_pred = np.array([2.8, 5.2, 7.5, 8.8, 10.7])

mse = calculate_mse(y_true, y_pred)
print(f"MSE: {mse:.4f}")

# 可视化真实值和预测值
plt.figure(figsize=(10, 6))
plt.plot(range(len(y_true)), y_true, 'bo', label='真实值')
plt.plot(range(len(y_pred)), y_pred, 'ro', label='预测值')
for i in range(len(y_true)):
    plt.plot([i, i], [y_true[i], y_pred[i]], 'k--', alpha=0.5)
plt.legend()
plt.title("真实值与预测值之间的差异")
plt.show()

## 3. 实现简单线性回归

在本节中，我们将从零开始实现简单线性回归，包括最小二乘法求解参数的代码实现。同时与sklearn的LinearRegression进行比较，并可视化回归结果。

### 3.1 最小二乘法推导

对于简单线性回归 $y = w_0 + w_1 x$，最小二乘法给出的参数解为：

$w_1 = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n} (x_i - \bar{x})^2}$

$w_0 = \bar{y} - w_1 \bar{x}$

其中 $\bar{x}$ 和 $\bar{y}$ 分别是x和y的平均值。

In [ ]:
# 从零实现简单线性回归
class SimpleLinearRegression:
    def __init__(self):
        self.w0 = None  # 截距
        self.w1 = None  # 斜率
        
    def fit(self, X, y):
        """使用最小二乘法拟合简单线性回归模型"""
        # 确保X是一维数组
        if X.ndim > 1:
            X = X.reshape(-1)
        
        # 计算均值
        x_mean = np.mean(X)
        y_mean = np.mean(y)
        
        # 计算分子和分母
        numerator = np.sum((X - x_mean) * (y - y_mean))
        denominator = np.sum((X - x_mean) ** 2)
        
        # 计算参数
        self.w1 = numerator / denominator
        self.w0 = y_mean - self.w1 * x_mean
        
        return self
    
    def predict(self, X):
        """使用拟合的模型进行预测"""
        # 确保X是一维数组
        if X.ndim > 1:
            X = X.reshape(-1)
        
        return self.w0 + self.w1 * X

In [ ]:
# 使用我们自己实现的简单线性回归模型
slr = SimpleLinearRegression()
slr.fit(X_simple.reshape(-1), y_simple)
y_pred_slr = slr.predict(X_simple.reshape(-1))

print(f"自实现模型参数: w0 = {slr.w0:.4f}, w1 = {slr.w1:.4f}")

# 使用sklearn的LinearRegression
lr = LinearRegression()
lr.fit(X_simple, y_simple)
y_pred_sklearn = lr.predict(X_simple)

print(f"Sklearn模型参数: w0 = {lr.intercept_:.4f}, w1 = {lr.coef_[0]:.4f}")

# 计算两个模型的MSE
mse_slr = calculate_mse(y_simple, y_pred_slr)
mse_sklearn = calculate_mse(y_simple, y_pred_sklearn)

print(f"自实现模型MSE: {mse_slr:.4f}")
print(f"Sklearn模型MSE: {mse_sklearn:.4f}")

In [ ]:
# 可视化结果
plt.figure(figsize=(12, 6))
plt.scatter(X_simple, y_simple, color='blue', alpha=0.7, label='数据点')
plt.plot(X_simple, y_pred_slr, color='red', linewidth=2, label=f'自实现模型 (w0={slr.w0:.2f}, w1={slr.w1:.2f})')
plt.plot(X_simple, y_pred_sklearn, color='green', linewidth=2, linestyle='--', 
         label=f'Sklearn模型 (w0={lr.intercept_:.2f}, w1={lr.coef_[0]:.2f})')
plt.xlabel('X')
plt.ylabel('y')
plt.title('简单线性回归：自实现模型 vs Sklearn模型')
plt.legend()
plt.show()

## 4. 多元线性回归

在本节中，我们将扩展到多元线性回归，使用矩阵运算进行实现。我们将展示如何处理多个特征，以及如何进行特征工程(标准化、归一化等)。

### 4.1 多元线性回归的矩阵表示

多元线性回归可以用矩阵形式表示为：

$\mathbf{y} = \mathbf{X} \mathbf{w} + \mathbf{\epsilon}$

其中：
- $\mathbf{y}$ 是 $n \times 1$ 目标值向量
- $\mathbf{X}$ 是 $n \times (p+1)$ 特征矩阵（包含一列1对应截距）
- $\mathbf{w}$ 是 $(p+1) \times 1$ 权重向量
- $\mathbf{\epsilon}$ 是 $n \times 1$ 误差向量

使用最小二乘法的闭式解为：

$\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$

In [ ]:
# 从零实现多元线性回归
class MultipleLinearRegression:
    def __init__(self):
        self.w = None  # 权重向量
        
    def fit(self, X, y):
        """使用最小二乘法拟合多元线性回归模型"""
        # 添加截距项（一列1）
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        
        # 计算最小二乘法的闭式解
        # w = (X^T X)^(-1) X^T y
        self.w = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
        
        return self
    
    def predict(self, X):
        """使用拟合的模型进行预测"""
        # 添加截距项
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        
        return X_b @ self.w

In [ ]:
# 将多变量数据集分为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

# 使用自实现的多元线性回归
mlr = MultipleLinearRegression()
mlr.fit(X_train, y_train)
y_pred_mlr = mlr.predict(X_test)

# 使用sklearn的LinearRegression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_sklearn = lr.predict(X_test)

# 计算两个模型的MSE
mse_mlr = calculate_mse(y_test, y_pred_mlr)
mse_sklearn = calculate_mse(y_test, y_pred_sklearn)

print(f"自实现多元线性回归模型MSE: {mse_mlr:.4f}")
print(f"Sklearn多元线性回归模型MSE: {mse_sklearn:.4f}")

# 打印模型参数
print("\n自实现模型参数:")
print(f"截距 (w0): {mlr.w[0]:.4f}")
print(f"特征权重: {mlr.w[1:]}")

print("\nSklearn模型参数:")
print(f"截距 (w0): {lr.intercept_:.4f}")
print(f"特征权重: {lr.coef_}")

### 4.2 特征工程 - 标准化

在实际应用中，特征的尺度差异可能很大，这会使得梯度下降算法收敛更加困难，并可能导致某些特征权重被夸大或缩小。标准化可以帮助解决这个问题。

In [ ]:
# 应用特征标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 使用自实现的多元线性回归在标准化数据上
mlr_scaled = MultipleLinearRegression()
mlr_scaled.fit(X_train_scaled, y_train)
y_pred_mlr_scaled = mlr_scaled.predict(X_test_scaled)

# 使用sklearn的LinearRegression在标准化数据上
lr_scaled = LinearRegression()
lr_scaled.fit(X_train_scaled, y_train)
y_pred_sklearn_scaled = lr_scaled.predict(X_test_scaled)

# 计算两个模型的MSE
mse_mlr_scaled = calculate_mse(y_test, y_pred_mlr_scaled)
mse_sklearn_scaled = calculate_mse(y_test, y_pred_sklearn_scaled)

print(f"标准化后的自实现多元线性回归模型MSE: {mse_mlr_scaled:.4f}")
print(f"标准化后的Sklearn多元线性回归模型MSE: {mse_sklearn_scaled:.4f}")

In [ ]:
# 比较标准化前后模型的权重分布
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.bar(range(1, len(mlr.w)), mlr.w[1:], alpha=0.7)
plt.title("未标准化数据的特征权重")
plt.xlabel("特征")
plt.ylabel("权重")

plt.subplot(1, 2, 2)
plt.bar(range(1, len(mlr_scaled.w)), mlr_scaled.w[1:], alpha=0.7)
plt.title("标准化数据的特征权重")
plt.xlabel("特征")
plt.ylabel("权重")

plt.tight_layout()
plt.show()

## 5. 模型评估

在本节中，我们将实现各种评估指标计算，包括MSE、RMSE、MAE、R²等。我们还将使用交叉验证评估模型性能，并展示如何解释模型结果。

In [ ]:
# 实现各种评估指标
def calculate_rmse(y_true, y_pred):
    """计算均方根误差(RMSE)"""
    return np.sqrt(mean_squared_error(y_true, y_pred))

def calculate_mae(y_true, y_pred):
    """计算平均绝对误差(MAE)"""
    return mean_absolute_error(y_true, y_pred)

def calculate_r2(y_true, y_pred):
    """计算决定系数(R²)"""
    return r2_score(y_true, y_pred)

def calculate_adjusted_r2(y_true, y_pred, n_features):
    """计算调整后的决定系数(Adjusted R²)"""
    n = len(y_true)
    r2 = calculate_r2(y_true, y_pred)
    return 1 - (1 - r2) * (n - 1) / (n - n_features - 1)

# 在测试集上评估模型
model_results = {
    'MSE': [mse_mlr, mse_sklearn, mse_mlr_scaled, mse_sklearn_scaled],
    'RMSE': [calculate_rmse(y_test, y_pred_mlr),
            calculate_rmse(y_test, y_pred_sklearn),
            calculate_rmse(y_test, y_pred_mlr_scaled),
            calculate_rmse(y_test, y_pred_sklearn_scaled)],
    'MAE': [calculate_mae(y_test, y_pred_mlr),
           calculate_mae(y_test, y_pred_sklearn),
           calculate_mae(y_test, y_pred_mlr_scaled),
           calculate_mae(y_test, y_pred_sklearn_scaled)],
    'R²': [calculate_r2(y_test, y_pred_mlr),
          calculate_r2(y_test, y_pred_sklearn),
          calculate_r2(y_test, y_pred_mlr_scaled),
          calculate_r2(y_test, y_pred_sklearn_scaled)],
    'Adjusted R²': [calculate_adjusted_r2(y_test, y_pred_mlr, X_test.shape[1]),
                  calculate_adjusted_r2(y_test, y_pred_sklearn, X_test.shape[1]),
                  calculate_adjusted_r2(y_test, y_pred_mlr_scaled, X_test.shape[1]),
                  calculate_adjusted_r2(y_test, y_pred_sklearn_scaled, X_test.shape[1])]
}

# 创建结果DataFrame
results_df = pd.DataFrame(model_results, 
                         index=['自实现模型', 'Sklearn模型', 
                               '自实现模型(标准化)', 'Sklearn模型(标准化)'])
results_df

### 5.1 残差分析

残差分析是评估线性回归模型假设是否满足的重要工具。

In [ ]:
# 计算残差
residuals = y_test - y_pred_sklearn

plt.figure(figsize=(14, 8))

# 残差图
plt.subplot(2, 2, 1)
plt.scatter(y_pred_sklearn, residuals, alpha=0.7)
plt.axhline(y=0, color='r', linestyle='-')
plt.xlabel('预测值')
plt.ylabel('残差')
plt.title('残差散点图')

# 残差直方图
plt.subplot(2, 2, 2)
plt.hist(residuals, bins=20, alpha=0.7)
plt.xlabel('残差')
plt.ylabel('频率')
plt.title('残差直方图')

# 残差QQ图
from scipy import stats
plt.subplot(2, 2, 3)
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('残差QQ图')

# 实际值vs预测值
plt.subplot(2, 2, 4)
plt.scatter(y_test, y_pred_sklearn, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('实际值')
plt.ylabel('预测值')
plt.title('实际值 vs 预测值')

plt.tight_layout()
plt.show()

### 5.2 交叉验证

使用交叉验证可以更好地评估模型的泛化能力。

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

# 设置交叉验证
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 在原始数据和标准化数据上分别进行交叉验证
cv_scores_original = cross_val_score(LinearRegression(), X_multi, y_multi, 
                                   scoring='neg_mean_squared_error', cv=kf)
cv_scores_scaled = cross_val_score(LinearRegression(), 
                                 StandardScaler().fit_transform(X_multi), y_multi, 
                                 scoring='neg_mean_squared_error', cv=kf)

# 转换为正MSE
cv_mse_original = -cv_scores_original
cv_mse_scaled = -cv_scores_scaled

# 输出结果
print("交叉验证结果 (MSE):")
print(f"原始数据 - 平均: {cv_mse_original.mean():.4f}, 标准差: {cv_mse_original.std():.4f}")
print(f"标准化数据 - 平均: {cv_mse_scaled.mean():.4f}, 标准差: {cv_mse_scaled.std():.4f}")

# 可视化交叉验证结果
plt.figure(figsize=(10, 6))
plt.boxplot([cv_mse_original, cv_mse_scaled], labels=['原始数据', '标准化数据'])
plt.title('交叉验证MSE分布')
plt.ylabel('MSE')
plt.show()

## 6. 正则化技术

在本节中，我们将实现Ridge(L2正则化)和Lasso(L1正则化)回归，比较不同正则化参数对模型的影响。通过代码展示如何防止过拟合。

### 6.1 岭回归 (Ridge Regression)

岭回归通过添加L2正则化项来防止过拟合：

$\min_w \sum_{i=1}^{n}(y_i - w^T x_i)^2 + \alpha \sum_{j=1}^{p} w_j^2$

其中 $\alpha$ 是正则化强度参数。

In [ ]:
# 实现岭回归
class RidgeRegression:
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # 正则化参数
        self.w = None  # 权重向量
        
    def fit(self, X, y):
        """使用岭回归公式拟合模型"""
        # 添加截距项
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        n_features = X_b.shape[1]
        
        # 创建单位矩阵，但第一个元素(截距)不正则化
        I = np.eye(n_features)
        I[0, 0] = 0
        
        # 岭回归闭式解: w = (X^T X + alpha*I)^(-1) X^T y
        self.w = np.linalg.inv(X_b.T @ X_b + self.alpha * I) @ X_b.T @ y
        
        return self
    
    def predict(self, X):
        """使用拟合的模型进行预测"""
        # 添加截距项
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        
        return X_b @ self.w

### 6.2 Lasso回归

Lasso回归使用L1正则化，促使某些特征权重变为零，从而实现特征选择：

$\min_w \sum_{i=1}^{n}(y_i - w^T x_i)^2 + \alpha \sum_{j=1}^{p} |w_j|$

In [ ]:
# Lasso回归没有简单的闭式解，通常需要迭代优化方法求解
# 这里我们使用sklearn的实现来比较不同正则化技术

# 在标准化数据上比较不同正则化方法
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
ridge_scores = []
lasso_scores = []

for alpha in alphas:
    # Ridge回归
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    ridge_pred = ridge.predict(X_test_scaled)
    ridge_scores.append(calculate_mse(y_test, ridge_pred))
    
    # Lasso回归
    lasso = Lasso(alpha=alpha)
    lasso.fit(X_train_scaled, y_train)
    lasso_pred = lasso.predict(X_test_scaled)
    lasso_scores.append(calculate_mse(y_test, lasso_pred))

# 可视化不同alpha值对模型性能的影响
plt.figure(figsize=(10, 6))
plt.semilogx(alphas, ridge_scores, 'b-o', label='Ridge回归')
plt.semilogx(alphas, lasso_scores, 'r-o', label='Lasso回归')
plt.xlabel('正则化参数 (alpha)')
plt.ylabel('MSE')
plt.title('正则化强度与模型性能的关系')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 比较不同正则化方法的特征权重
plt.figure(figsize=(14, 6))

# 训练不同模型
lr = LinearRegression().fit(X_train_scaled, y_train)
ridge = Ridge(alpha=1.0).fit(X_train_scaled, y_train)
lasso = Lasso(alpha=0.1).fit(X_train_scaled, y_train)

# 获取特征权重
coef_lr = lr.coef_
coef_ridge = ridge.coef_
coef_lasso = lasso.coef_

# 创建特征索引
features = range(len(coef_lr))

# 绘制权重比较图
plt.subplot(1, 2, 1)
plt.bar(features, coef_lr, alpha=0.7, label='线性回归')
plt.bar(features, coef_ridge, alpha=0.5, label='Ridge (alpha=1.0)')
plt.xlabel('特征')
plt.ylabel('权重')
plt.title('线性回归 vs Ridge回归 权重比较')
plt.legend()

plt.subplot(1, 2, 2)
plt.bar(features, coef_lr, alpha=0.7, label='线性回归')
plt.bar(features, coef_lasso, alpha=0.5, label='Lasso (alpha=0.1)')
plt.xlabel('特征')
plt.ylabel('权重')
plt.title('线性回归 vs Lasso回归 权重比较')
plt.legend()

plt.tight_layout()
plt.show()

# 输出每个特征被置为零的数量 (Lasso的特征选择效果)
print(f"Lasso回归 (alpha=0.1) 将 {np.sum(coef_lasso == 0)} 个特征的权重置为零")

## 7. 梯度下降优化

在本节中，我们将实现梯度下降算法求解线性回归，包括批量梯度下降、随机梯度下降和小批量梯度下降。并可视化梯度下降过程。

### 7.1 批量梯度下降 (Batch Gradient Descent)

批量梯度下降在每次迭代时使用所有训练样本计算梯度。对于线性回归，权重更新规则为：

$w_j := w_j - \alpha \frac{1}{m}\sum_{i=1}^{m}(h_w(x^{(i)}) - y^{(i)})x_j^{(i)}$

其中 $\alpha$ 是学习率，$h_w(x^{(i)})$ 是模型对第i个样本的预测。

In [ ]:
# 实现带有梯度下降优化的线性回归
class LinearRegressionGD:
    def __init__(self, learning_rate=0.01, n_iterations=1000, batch_size=None):
        self.learning_rate = learning_rate  # 学习率
        self.n_iterations = n_iterations  # 迭代次数
        self.batch_size = batch_size  # 批量大小，None表示批量梯度下降
        self.w = None  # 权重向量
        self.cost_history = []  # 记录每次迭代的成本
        
    def fit(self, X, y):
        """使用梯度下降法拟合线性回归模型"""
        # 添加截距项
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        n_samples, n_features = X_b.shape
        
        # 初始化权重
        self.w = np.random.randn(n_features)
        
        # 梯度下降迭代
        for i in range(self.n_iterations):
            # 确定当前批次的索引
            if self.batch_size is None:  # 批量梯度下降
                indices = np.arange(n_samples)
            else:  # 小批量或随机梯度下降
                indices = np.random.choice(n_samples, self.batch_size, replace=False)
            
            X_batch = X_b[indices]
            y_batch = y[indices]
            
            # 计算预测
            y_pred = X_batch @ self.w
            
            # 计算梯度
            gradients = 2/len(y_batch) * X_batch.T @ (y_pred - y_batch)
            
            # 更新权重
            self.w = self.w - self.learning_rate * gradients
            
            # 记录成本
            cost = np.mean((X_b @ self.w - y) ** 2)
            self.cost_history.append(cost)
        
        return self
    
    def predict(self, X):
        """使用拟合的模型进行预测"""
        # 添加截距项
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        
        return X_b @ self.w

In [ ]:
# 在简单的一元线性回归数据上测试不同的梯度下降方法
# 创建训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X_simple, y_simple, test_size=0.2, random_state=42)

# 训练不同的梯度下降模型
batch_gd = LinearRegressionGD(learning_rate=0.01, n_iterations=1000, batch_size=None)
batch_gd.fit(X_train, y_train)

sgd = LinearRegressionGD(learning_rate=0.01, n_iterations=1000, batch_size=1)
sgd.fit(X_train, y_train)

mini_batch_gd = LinearRegressionGD(learning_rate=0.01, n_iterations=1000, batch_size=16)
mini_batch_gd.fit(X_train, y_train)

# 计算预测结果
y_pred_batch = batch_gd.predict(X_test)
y_pred_sgd = sgd.predict(X_test)
y_pred_mini = mini_batch_gd.predict(X_test)

# 计算MSE
mse_batch = calculate_mse(y_test, y_pred_batch)
mse_sgd = calculate_mse(y_test, y_pred_sgd)
mse_mini = calculate_mse(y_test, y_pred_mini)

print(f"批量梯度下降MSE: {mse_batch:.4f}")
print(f"随机梯度下降MSE: {mse_sgd:.4f}")
print(f"小批量梯度下降MSE: {mse_mini:.4f}")

In [ ]:
# 可视化成本历史
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(batch_gd.cost_history, label='批量梯度下降')
plt.plot(sgd.cost_history, label='随机梯度下降')
plt.plot(mini_batch_gd.cost_history, label='小批量梯度下降')
plt.xlabel('迭代次数')
plt.ylabel('成本 (MSE)')
plt.title('梯度下降成本历史')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(batch_gd.cost_history[:100], label='批量梯度下降')
plt.plot(sgd.cost_history[:100], label='随机梯度下降')
plt.plot(mini_batch_gd.cost_history[:100], label='小批量梯度下降')
plt.xlabel('迭代次数')
plt.ylabel('成本 (MSE)')
plt.title('前100次迭代的成本历史')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 可视化不同梯度下降方法的回归线
plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, alpha=0.7, label='测试数据')
plt.plot(X_test, y_pred_batch, color='red', linewidth=2, 
         label=f'批量梯度下降 (MSE={mse_batch:.2f})')
plt.plot(X_test, y_pred_sgd, color='green', linewidth=2, 
         label=f'随机梯度下降 (MSE={mse_sgd:.2f})')
plt.plot(X_test, y_pred_mini, color='blue', linewidth=2, 
         label=f'小批量梯度下降 (MSE={mse_mini:.2f})')
plt.xlabel('X')
plt.ylabel('y')
plt.title('不同梯度下降方法的线性回归结果')
plt.legend()
plt.grid(True)
plt.show()

### 7.2 学习率对梯度下降的影响

学习率是梯度下降算法中最重要的超参数之一。我们将探索不同学习率对算法收敛的影响。

In [ ]:
# 测试不同学习率
learning_rates = [0.0001, 0.001, 0.01, 0.1, 0.5]
models = []

for lr in learning_rates:
    model = LinearRegressionGD(learning_rate=lr, n_iterations=200, batch_size=None)
    model.fit(X_train, y_train)
    models.append(model)

# 可视化不同学习率的成本历史
plt.figure(figsize=(12, 6))
for i, lr in enumerate(learning_rates):
    plt.plot(models[i].cost_history, label=f'学习率 = {lr}')
plt.xlabel('迭代次数')
plt.ylabel('成本 (MSE)')
plt.title('不同学习率的梯度下降收敛情况')
plt.legend()
plt.grid(True)
plt.show()

## 8. 实际应用案例

在本节中，我们将使用California Housing数据集应用线性回归模型，完成从数据预处理、特征选择到模型训练、评估的完整流程。

In [ ]:
# 探索California Housing数据集
print(f"California Housing数据集形状: {X_housing.shape}")
print(f"特征名称: {X_housing.columns.tolist()}")

# 数据简要统计
print("\n数据统计摘要:")
X_housing.describe()

In [ ]:
# 可视化目标变量分布
plt.figure(figsize=(10, 6))
plt.hist(y_housing, bins=40, alpha=0.7)
plt.xlabel('房价中位数 (单位: $100,000)')
plt.ylabel('频率')
plt.title('California房价中位数分布')
plt.grid(True)
plt.show()

In [ ]:
# 探索特征之间的相关性
plt.figure(figsize=(12, 10))
correlation_matrix = X_housing.copy()
correlation_matrix['MedHouseVal'] = y_housing
sns.heatmap(correlation_matrix.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('特征相关性矩阵')
plt.show()

In [ ]:
# 查看每个特征与目标变量之间的散点图
plt.figure(figsize=(15, 10))
for i, feature in enumerate(X_housing.columns):
    plt.subplot(3, 3, i+1)
    plt.scatter(X_housing[feature], y_housing, alpha=0.1)
    plt.xlabel(feature)
    plt.ylabel('房价中位数')
    plt.title(f'{feature} vs 房价')
plt.tight_layout()
plt.show()

In [ ]:
# 准备数据
X_train, X_test, y_train, y_test = train_test_split(X_housing, y_housing, test_size=0.2, random_state=42)

# 特征标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 创建并训练模型
models = {
    "线性回归": LinearRegression(),
    "岭回归 (alpha=1.0)": Ridge(alpha=1.0),
    "Lasso回归 (alpha=0.01)": Lasso(alpha=0.01)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    # 计算评估指标
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R²": r2
    }
    
    # 输出特征权重
    if hasattr(model, 'coef_'):
        coefs = pd.Series(model.coef_, index=X_housing.columns)
        print(f"\n{name} 特征权重:")
        print(coefs.sort_values(ascending=False))
    
# 创建结果DataFrame
results_df = pd.DataFrame(results).T
results_df

In [ ]:
# 可视化实际值与预测值的比较
plt.figure(figsize=(16, 5))
for i, (name, model) in enumerate(models.items()):
    plt.subplot(1, 3, i+1)
    y_pred = model.predict(X_test_scaled)
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.xlabel('实际值')
    plt.ylabel('预测值')
    plt.title(f'{name}\nR² = {r2_score(y_test, y_pred):.3f}')
plt.tight_layout()
plt.show()

In [ ]:
# 可视化实际房价与预测房价的地理分布
# 注意：需要有longitude和latitude特征才能执行此可视化
if 'Longitude' in X_housing and 'Latitude' in X_housing:
    best_model_name = results_df['R²'].idxmax()
    best_model = models[best_model_name]
    
    # 预测整个数据集
    X_full_scaled = scaler.transform(X_housing)
    y_pred_full = best_model.predict(X_full_scaled)
    
    # 计算预测误差
    error = y_housing - y_pred_full
    
    # 创建地理可视化
    plt.figure(figsize=(15, 12))
    
    # 实际房价分布
    plt.subplot(2, 1, 1)
    plt.scatter(X_housing['Longitude'], X_housing['Latitude'], 
               c=y_housing, cmap='viridis', alpha=0.5, s=10)
    plt.colorbar(label='实际房价中位数')
    plt.title('California房价地理分布（实际值）')
    plt.xlabel('经度')
    plt.ylabel('纬度')
    
    # 预测误差分布
    plt.subplot(2, 1, 2)
    sc = plt.scatter(X_housing['Longitude'], X_housing['Latitude'], 
                    c=error, cmap='coolwarm', alpha=0.5, s=10)
    plt.colorbar(sc, label='预测误差')
    plt.title(f'预测误差地理分布（使用{best_model_name}）')
    plt.xlabel('经度')
    plt.ylabel('纬度')
    
    plt.tight_layout()
    plt.show()
else:
    print("数据集中没有经度和纬度特征，无法创建地理可视化。")

## 总结

在本教程中，我们从零开始实现了线性回归算法，并学习了：

1. **线性回归的基础知识**，包括模型表达式和损失函数。
2. **如何从零实现简单线性回归和多元线性回归**，使用最小二乘法的闭式解。
3. **特征工程的重要性**，特别是标准化对模型性能的影响。
4. **如何评估模型性能**，使用各种指标和交叉验证。
5. **正则化技术**，包括Ridge回归和Lasso回归，以及它们如何帮助解决过拟合问题。
6. **梯度下降优化**，包括批量梯度下降、随机梯度下降和小批量梯度下降。
7. **实际应用案例**，使用California Housing数据集应用线性回归。

线性回归是更复杂机器学习和深度学习模型的基础。掌握线性回归的原理和实现，对于理解更高级的算法和模型至关重要。